In [2]:
#!/usr/bin/env python3
"""
Generate QE input files for strained BiSb monolayers.
No external libraries — stdlib only.
Usage: python3 generate_qe_inputs.py
"""

import os
import math

# -------------------------------------------------------
# BASE LATTICE (same as your VASP POSCAR, in Angstrom)
# -------------------------------------------------------
BASE_AX = 4.2620340227358220   # a1x
BASE_BX = -2.1310170113679110  # a2x
BASE_BY =  3.6910297357198201  # a2y
BASE_CZ = 18.1184253028625299  # vacuum axis (z)

STRAINS = [2,4,6,8,10,12,14,16,18]   # percent — extend freely

# Pseudopotential filenames (ONCV or PAW-style USPP, your choice)
PP_BI = "Bi.upf"
PP_SB = "Sb.upf"
PSEUDO_DIR = "/arf/home/rkoc/pseudo"

# VASP used vasp_ncl → we need noncollinear + SOC in QE too
# lspinorb = .true. requires fully-relativistic PPs

# -------------------------------------------------------
# HELPER: Angstrom → Bohr  (QE uses atomic units internally
# but 'angstrom' card lets us stay in Å — we use that)
# -------------------------------------------------------
ANG2BOHR = 1.8897259886  # only needed if you ever want bohr explicitly

def strain_lattice(p):
    """Return in-plane lattice vectors scaled by (1 + p/100)."""
    f = 1.0 + p / 100.0
    return f * BASE_AX, f * BASE_BX, f * BASE_BY


def write_relax(dirpath, p, ax, bx, by):
    """vc-relax with fixed cell shape — only atomic positions relax.
    For a monolayer under fixed biaxial strain you typically want
    relax (not vc-relax) so the cell stays pinned. Adjust as needed."""
    content = f"""&CONTROL
  calculation   = 'relax'
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  pseudo_dir    = '{PSEUDO_DIR}'
  tprnfor       = .true.
  tstress       = .true.
/
&SYSTEM
  ibrav         = 0
  nat           = 2
  ntyp          = 2
  ecutwfc       = 80
  ecutrho       = 640
  occupations   = 'smearing'
  smearing      = 'mp'
  degauss       = 0.01
  noncolin      = .true.
  lspinorb      = .true.
/
&ELECTRONS
  conv_thr      = 1.0d-8
  mixing_beta   = 0.4
/
&IONS
  ion_dynamics  = 'bfgs'
/
ATOMIC_SPECIES
  Bi  208.980  {PP_BI}
  Sb  121.760  {PP_SB}

K_POINTS automatic
  9 9 1  0 0 0

CELL_PARAMETERS angstrom
  {ax:.10f}   0.0000000000   0.0000000000
  {bx:.10f}   {by:.10f}   0.0000000000
  0.0000000000   0.0000000000   {BASE_CZ:.10f}

ATOMIC_POSITIONS crystal
  Bi   0.000000000   0.000000000   0.546550649
  Sb   0.333333333   0.666666666   0.453449351
"""
    with open(os.path.join(dirpath, "relax.in"), "w") as f:
        f.write(content)


def write_scf(dirpath, p, ax, bx, by):
    content = f"""&CONTROL
  calculation   = 'scf'
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  pseudo_dir    = '{PSEUDO_DIR}'
  tprnfor       = .true.
/
&SYSTEM
  ibrav         = 0
  nat           = 2
  ntyp          = 2
  ecutwfc       = 80
  ecutrho       = 640
  occupations   = 'smearing'
  smearing      = 'mp'
  degauss       = 0.01
  noncolin      = .true.
  lspinorb      = .true.
/
&ELECTRONS
  conv_thr      = 1.0d-8
  mixing_beta   = 0.4
/
ATOMIC_SPECIES
  Bi  208.980  {PP_BI}
  Sb  121.760  {PP_SB}

K_POINTS automatic
  9 9 1  0 0 0

CELL_PARAMETERS angstrom
  {ax:.10f}   0.0000000000   0.0000000000
  {bx:.10f}   {by:.10f}   0.0000000000
  0.0000000000   0.0000000000   {BASE_CZ:.10f}

ATOMIC_POSITIONS crystal
  Bi   0.000000000   0.000000000   0.546550649
  Sb   0.333333333   0.666666666   0.453449351
"""
    with open(os.path.join(dirpath, "scf.in"), "w") as f:
        f.write(content)


def write_nscf(dirpath, p, ax, bx, by):
    """Dense k-mesh NSCF for DOS or Wannier projection."""
    content = f"""&CONTROL
  calculation   = 'nscf'
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  pseudo_dir    = '{PSEUDO_DIR}'
/
&SYSTEM
  ibrav         = 0
  nat           = 2
  ntyp          = 2
  ecutwfc       = 80
  ecutrho       = 640
  occupations   = 'tetrahedra_opt'
  noncolin      = .true.
  lspinorb      = .true.
/
&ELECTRONS
  conv_thr      = 1.0d-10
/
ATOMIC_SPECIES
  Bi  208.980  {PP_BI}
  Sb  121.760  {PP_SB}

K_POINTS automatic
  18 18 1  0 0 0

CELL_PARAMETERS angstrom
  {ax:.10f}   0.0000000000   0.0000000000
  {bx:.10f}   {by:.10f}   0.0000000000
  0.0000000000   0.0000000000   {BASE_CZ:.10f}

ATOMIC_POSITIONS crystal
  Bi   0.000000000   0.000000000   0.546550649
  Sb   0.333333333   0.666666666   0.453449351
"""
    with open(os.path.join(dirpath, "nscf.in"), "w") as f:
        f.write(content)


# -------------------------------------------------------
# K-PATH:  Γ → M → K → Γ  for hexagonal 2D
# -------------------------------------------------------
KPATH_POINTS = [
    ("G",  0.0000, 0.0000, 0.0),
    ("M",  0.5000, 0.0000, 0.0),
    ("K",  0.3333, 0.3333, 0.0),
    ("G",  0.0000, 0.0000, 0.0),
]
N_BANDS_BETWEEN = 40   # k-points between each high-sym pair

def build_kpath():
    """Return the K_POINTS crystal_b block string."""
    lines = [f"K_POINTS crystal_b", f"{len(KPATH_POINTS)}"]
    for label, kx, ky, kz in KPATH_POINTS:
        lines.append(f"  {kx:.6f}  {ky:.6f}  {kz:.6f}  {N_BANDS_BETWEEN}  ! {label}")
    return "\n".join(lines)


def write_bands(dirpath, p, ax, bx, by):
    kblock = build_kpath()
    content = f"""&CONTROL
  calculation   = 'bands'
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  pseudo_dir    = '{PSEUDO_DIR}'
/
&SYSTEM
  ibrav         = 0
  nat           = 2
  ntyp          = 2
  ecutwfc       = 80
  ecutrho       = 640
  occupations   = 'smearing'
  smearing      = 'mp'
  degauss       = 0.01
  noncolin      = .true.
  lspinorb      = .true.
  nbnd          = 40
/
&ELECTRONS
  conv_thr      = 1.0d-10
/
ATOMIC_SPECIES
  Bi  208.980  {PP_BI}
  Sb  121.760  {PP_SB}

{kblock}

CELL_PARAMETERS angstrom
  {ax:.10f}   0.0000000000   0.0000000000
  {bx:.10f}   {by:.10f}   0.0000000000
  0.0000000000   0.0000000000   {BASE_CZ:.10f}

ATOMIC_POSITIONS crystal
  Bi   0.000000000   0.000000000   0.546550649
  Sb   0.333333333   0.666666666   0.453449351
"""
    with open(os.path.join(dirpath, "bands.in"), "w") as f:
        f.write(content)


def write_bands_pp(dirpath, p):
    """bands.x post-processing input."""
    content = f"""&BANDS
  prefix      = 'BiSb_{p}pct'
  outdir      = './out'
  filband     = 'BiSb_{p}pct.bands.dat'
  spin_component = 0
  lsym        = .false.
/
"""
    with open(os.path.join(dirpath, "bands_pp.in"), "w") as f:
        f.write(content)


# -------------------------------------------------------
# MAIN
# -------------------------------------------------------
def main():
    for p in STRAINS:
        ax, bx, by = strain_lattice(p)
        dirpath = f"BiSb_strain_{p}_percent"
        os.makedirs(dirpath, exist_ok=True)
        os.makedirs(os.path.join(dirpath, "out"), exist_ok=True)

        write_relax(dirpath, p, ax, bx, by)
        write_scf(dirpath, p, ax, bx, by)
        write_nscf(dirpath, p, ax, bx, by)
        write_bands(dirpath, p, ax, bx, by)
        write_bands_pp(dirpath, p)

        print(f"[strain {p:>2}%]  a = {ax:.8f} Å   dir → {dirpath}/")

    print("\nAll input files generated.")


if __name__ == "__main__":
    main()

[strain  2%]  a = 4.34727470 Å   dir → BiSb_strain_2_percent/
[strain  4%]  a = 4.43251538 Å   dir → BiSb_strain_4_percent/
[strain  6%]  a = 4.51775606 Å   dir → BiSb_strain_6_percent/
[strain  8%]  a = 4.60299674 Å   dir → BiSb_strain_8_percent/
[strain 10%]  a = 4.68823743 Å   dir → BiSb_strain_10_percent/
[strain 12%]  a = 4.77347811 Å   dir → BiSb_strain_12_percent/
[strain 14%]  a = 4.85871879 Å   dir → BiSb_strain_14_percent/
[strain 16%]  a = 4.94395947 Å   dir → BiSb_strain_16_percent/
[strain 18%]  a = 5.02920015 Å   dir → BiSb_strain_18_percent/

All input files generated.


In [3]:
#!/usr/bin/env python3
"""
Generate QE + Wannier90 input files for strained BiSb monolayers.
stdlib + subprocess only (for kmesh.pl).
"""

import os
import subprocess

# -------------------------------------------------------
# PARAMETERS
# -------------------------------------------------------
BASE_AX =  4.2620340227358220
BASE_BX = -2.1310170113679110
BASE_BY =  3.6910297357198201
BASE_CZ = 18.1184253028625299

STRAINS    = [9, 11, 13]
PP_BI      = "Bi.upf"
PP_SB      = "Sb.upf"
PSEUDO_DIR = "/arf/home/rkoc/pseudo"

# Wannier90 parameters — tune these to your system
NUM_WANN   = 16    # 8 from Bi (6p+2s), 8 from Sb — adjust after disentanglement test
NUM_BANDS  = 40
DIS_WIN_MAX  =  4.0   # eV above Fermi — adjust after first run
DIS_WIN_MIN  = -6.0
DIS_FROZ_MAX =  1.5
DIS_FROZ_MIN = -4.0

# K-mesh for NSCF/Wannier — must be the same
NK1, NK2, NK3 = 9, 9, 1


# -------------------------------------------------------
# HELPERS
# -------------------------------------------------------
def strain_lattice(p):
    f = 1.0 + p / 100.0
    return f * BASE_AX, f * BASE_BX, f * BASE_BY


def get_kpoints_block(nk1, nk2, nk3):
    """
    Call kmesh.pl and return its output as a string.
    kmesh.pl prints the K_POINTS crystal block directly.
    Falls back to a clear error if kmesh.pl is not on PATH.
    """
    try:
        result = subprocess.run(
            ["kmesh.pl", str(nk1), str(nk2), str(nk3)],
            capture_output=True, text=True, check=True
        )
        return result.stdout.strip()
    except FileNotFoundError:
        raise RuntimeError(
            "kmesh.pl not found on PATH. "
            "Source your Wannier90 environment before running this script."
        )


def cell_parameters_block(ax, bx, by):
    return (
        f"CELL_PARAMETERS angstrom\n"
        f"  {ax:.10f}   0.0000000000   0.0000000000\n"
        f"  {bx:.10f}   {by:.10f}   0.0000000000\n"
        f"  0.0000000000   0.0000000000   {BASE_CZ:.10f}"
    )


ATOMIC_SPECIES_BLOCK = f"""ATOMIC_SPECIES
  Bi  208.980  {PP_BI}
  Sb  121.760  {PP_SB}"""

ATOMIC_POSITIONS_BLOCK = """ATOMIC_POSITIONS crystal
  Bi   0.000000000   0.000000000   0.546550649
  Sb   0.333333333   0.666666666   0.453449351"""

COMMON_SYSTEM = f"""  ibrav         = 0
  nat           = 2
  ntyp          = 2
  ecutwfc       = 80
  ecutrho       = 640
  noncolin      = .true.
  lspinorb      = .true.
  nbnd          = {NUM_BANDS}"""


# -------------------------------------------------------
# pw.x NSCF  (explicit k-mesh via kmesh.pl)
# -------------------------------------------------------
def write_nscf(dirpath, p, ax, bx, by, kblock):
    content = f"""&CONTROL
  calculation   = 'nscf'
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  pseudo_dir    = '{PSEUDO_DIR}'
/
&SYSTEM
{COMMON_SYSTEM}
  occupations   = 'tetrahedra_opt'
/
&ELECTRONS
  conv_thr      = 1.0d-10
  diago_full_acc = .true.
/
{ATOMIC_SPECIES_BLOCK}

{kblock}

{cell_parameters_block(ax, bx, by)}

{ATOMIC_POSITIONS_BLOCK}
"""
    _write(dirpath, "nscf.in", content)


# -------------------------------------------------------
# pw2wannier90.x input
# -------------------------------------------------------
def write_pw2wan(dirpath, p):
    """
    spin_component is irrelevant for noncolin — omit it.
    write_amn/mmn/eig are all needed by wannier90.
    write_uHu is optional but useful for orbital magnetization.
    """
    content = f"""&INPUTPP
  prefix        = 'BiSb_{p}pct'
  outdir        = './out'
  seedname      = 'wannier90'
  write_mmn     = .true.
  write_amn     = .true.
  write_eig     = .true.
  write_unk     = .false.
/
"""
    _write(dirpath, "pw2wan.in", content)


# -------------------------------------------------------
# wannier90.win
# -------------------------------------------------------
def write_wannier90_win(dirpath, p, ax, bx, by, kblock):
    """
    Projections for buckled BiSb (space group P3m1):
      Bi sits at (0,0,0.547) — p orbitals dominate near Fermi
      Sb sits at (1/3,2/3,0.453) — same
    With SOC and noncolin, W90 uses spinor projections:
    each orbital gets spin-up and spin-down → 2 × 8 = 16 wannier functions
    for a minimal p-only basis (px,py,pz × 2 atoms × 2 spins).

    The k_points block must be identical to the NSCF k-mesh —
    we reuse the same kblock from kmesh.pl, but strip the
    'K_POINTS crystal' header line since W90 uses its own format.
    """
    # kmesh.pl output looks like:
    #   K_POINTS crystal
    #   81
    #   0.00000  0.00000  0.00000  0.01235
    #   ...
    # W90 wants just the number and the coordinate lines (no weight column,
    # or with weight — W90 ignores weights). We strip the header.
    klines = kblock.split("\n")
    # drop "K_POINTS crystal" line, keep count + coordinate lines
    w90_klines = [l for l in klines if not l.strip().startswith("K_POINTS")]
    w90_kblock = "\n".join(w90_klines)

    # Reciprocal lattice vectors (in units of 2π/Å) — W90 needs them
    # For hexagonal: b1 = (1/a, 1/√3·a, 0), b2 = (0, 2/√3·a, 0) (in Å⁻¹)
    # We write them in Cartesian Å⁻¹ × (2π) = W90 convention
    import math
    b1x =  2 * math.pi / ax
    b2x = -2 * math.pi / (ax * math.sqrt(3))  # approximate for hexagonal
    b2y =  2 * 2 * math.pi / (ax * math.sqrt(3))
    # Note: for accurate reciprocal vectors you should compute from the
    # actual strained cell. These are approximate; W90 also accepts
    # unit_cell_cart + atoms_cart and derives them itself (preferred).

    # Compute Cartesian atomic positions (Å)
    bi_cart_x = 0.0
    bi_cart_y = 0.0
    bi_cart_z = 0.546550649 * BASE_CZ
    sb_cart_x = (1/3) * ax + (2/3) * bx
    sb_cart_y = (2/3) * by
    sb_cart_z = 0.453449351 * BASE_CZ

    content = f"""! Wannier90 input for BiSb strain {p}%
! Noncollinear + SOC: spinor_projections must be .true.

num_wann        = {NUM_WANN}
num_bands       = {NUM_BANDS}
spinors         = .true.

! --- Disentanglement windows (eV, relative to Fermi) ---
! These are starting guesses — adjust after inspecting the band structure
dis_win_max     =  {DIS_WIN_MAX}
dis_win_min     =  {DIS_WIN_MIN}
dis_froz_max    =  {DIS_FROZ_MAX}
dis_froz_min    =  {DIS_FROZ_MIN}
dis_num_iter    =  500
dis_conv_tol    =  1.0d-10

! --- Wannierisation ---
num_iter        =  500
conv_tol        =  1.0d-10

! --- Output ---
write_hr        = .true.
write_tb        = .true.
write_xyz       = .true.
bands_plot      = .true.

! --- Band path for interpolated bands (Γ-M-K-Γ) ---
bands_num_points = 100
begin kpoint_path
G  0.0000  0.0000  0.0000   M  0.5000  0.0000  0.0000
M  0.5000  0.0000  0.0000   K  0.3333  0.3333  0.0000
K  0.3333  0.3333  0.0000   G  0.0000  0.0000  0.0000
end kpoint_path

! --- Projections (spinor basis) ---
! px,py,pz on each site; W90 doubles them for spinors automatically
begin projections
Bi : p
Sb : p
end projections

! --- Unit cell (Å) ---
begin unit_cell_cart
angstrom
  {ax:.10f}   0.0000000000   0.0000000000
  {bx:.10f}   {by:.10f}   0.0000000000
  0.0000000000   0.0000000000   {BASE_CZ:.10f}
end unit_cell_cart

! --- Atomic positions (fractional) ---
begin atoms_frac
Bi   0.000000000   0.000000000   0.546550649
Sb   0.333333333   0.666666666   0.453449351
end atoms_frac

! --- K-points (from kmesh.pl {NK1}x{NK2}x{NK3}) ---
mp_grid : {NK1} {NK2} {NK3}

begin kpoints
{w90_kblock}
end kpoints
"""
    _write(dirpath, "wannier90.win", content)


# -------------------------------------------------------
# UTILITY
# -------------------------------------------------------
def _write(dirpath, filename, content):
    path = os.path.join(dirpath, filename)
    with open(path, "w") as f:
        f.write(content)
    print(f"  wrote {path}")


# -------------------------------------------------------
# MAIN
# -------------------------------------------------------
def main():
    # Get k-points once — same mesh for all strains
    print(f"Calling kmesh.pl {NK1} {NK2} {NK3} ...")
    kblock = get_kpoints_block(NK1, NK2, NK3)
    print(f"  → {kblock.count(chr(10))} k-points lines captured.\n")

    for p in STRAINS:
        ax, bx, by = strain_lattice(p)
        dirpath = f"BiSb_strain_{p}_percent"
        os.makedirs(os.path.join(dirpath, "out"), exist_ok=True)

        print(f"[strain {p:>2}%]  a = {ax:.8f} Å")
        write_nscf(dirpath, p, ax, bx, by, kblock)
        write_pw2wan(dirpath, p)
        write_wannier90_win(dirpath, p, ax, bx, by, kblock)

    print("\nAll files generated.")


if __name__ == "__main__":
    main()

Calling kmesh.pl 9 9 1 ...


RuntimeError: kmesh.pl not found on PATH. Source your Wannier90 environment before running this script.